# CHSA Medical Triage Agent - Kaggle Experiments

This notebook runs tracked 5k or 8k SFT+DPO experiments on a Kaggle GPU without reinstalling Kaggle's built-in PyTorch stack. It downloads the selected private Hugging Face dataset, audits it locally, starts an MLflow UI through ngrok, then runs the experiment orchestrator.

Before running: enable a Kaggle GPU and add Kaggle secrets named `HF_TOKEN` and `NGROK_AUTHTOKEN`. `HF_TOKEN` must have access to `Lokhidor/medical-triage-dataset` and `Lokhidor/medical-triage-dataset-8k`.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Nhkp/medical-triage-agent.git"
REPO_DIR = Path("/kaggle/working/medical-triage-agent")

if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a git checkout; remove it or choose another path"
        )
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(Path.cwd())

## Install dependencies without reinstalling Torch

Kaggle already ships PyTorch. Installing a second `torch` wheel into `.venv` can exhaust disk space, so this notebook uses the system environment and installs only the missing project dependencies.

In [ ]:
!python -m pip install -q -U \
  datasets peft trl transformers accelerate bitsandbytes pyyaml mlflow pyngrok wrapt

In [ ]:
import torch

print("cuda_available=", torch.cuda.is_available())
print("device=", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("torch=", torch.__version__)

## Configure credentials and experiment

Choose `DATASET_SIZE = "5k"` or `DATASET_SIZE = "8k"`. Keep the smoke values small first, then increase them or set them to `None` for a full run.

In [ ]:
from kaggle_secrets import UserSecretsClient

DATASET_SIZE = "8k"  # "5k" or "8k"
MAX_STEPS = 5
MAX_TRAIN_SAMPLES = 32
PUSH_TO_HUB = False
MLFLOW_UI_PORT = 5000

DATASETS = {
    "5k": {
        "label": "dataset-5k",
        "repo": "Lokhidor/medical-triage-dataset",
        "local_dir": "data/processed/training-5k",
        "sft_repo": "Lokhidor/medical-triage-qwen3-sft-lora-5k",
        "dpo_repo": "Lokhidor/medical-triage-qwen3-dpo-lora-5k",
    },
    "8k": {
        "label": "dataset-8k",
        "repo": "Lokhidor/medical-triage-dataset-8k",
        "local_dir": "data/processed/training-8000",
        "sft_repo": "Lokhidor/medical-triage-qwen3-sft-lora-8k",
        "dpo_repo": "Lokhidor/medical-triage-qwen3-dpo-lora-8k",
    },
}

if DATASET_SIZE not in DATASETS:
    raise ValueError("DATASET_SIZE must be '5k' or '8k'")

settings = DATASETS[DATASET_SIZE]
secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["NGROK_AUTHTOKEN"] = secrets.get_secret("NGROK_AUTHTOKEN")
os.environ["HF_DATASET_REPO"] = settings["repo"]
os.environ["HF_SFT_MODEL_REPO"] = settings["sft_repo"]
os.environ["HF_DPO_MODEL_REPO"] = settings["dpo_repo"]

print("HF token loaded:", bool(os.environ.get("HF_TOKEN")))
print("ngrok token loaded:", bool(os.environ.get("NGROK_AUTHTOKEN")))
print("Dataset label:", settings["label"])
print("Dataset repo:", os.environ["HF_DATASET_REPO"])

In [ ]:
!hf auth whoami

## Download and audit the selected dataset

Each dataset size is kept in its own local folder so 5k and 8k runs do not overwrite each other.

In [ ]:
local_dir = settings["local_dir"]
!rm -rf "$local_dir"
!mkdir -p "$local_dir"
!hf download "$HF_DATASET_REPO" \
  --type dataset \
  --local-dir "$local_dir" \
  --include "sft_*.jsonl" \
  --include "dpo_*.jsonl" \
  --include "manifest.json" \
  --include "README.md" \
  --include "audit_report.json"

In [ ]:
!PYTHONPATH=src python -m medical_triage_agent audit-training-data "$local_dir"
!PYTHONPATH=src python -m medical_triage_agent summarize-training-data "$local_dir"

## Dry-run the tracked SFT+DPO experiment

This validates paths, config overrides, output directories, and the MLflow UI command without loading model weights.

In [ ]:
max_steps_arg = [] if MAX_STEPS is None else ["--max-steps", str(MAX_STEPS)]
max_samples_arg = (
    [] if MAX_TRAIN_SAMPLES is None else ["--max-train-samples", str(MAX_TRAIN_SAMPLES)]
)
push_args = (
    [
        "--push-to-hub",
        "--sft-hub-model-id",
        settings["sft_repo"],
        "--dpo-hub-model-id",
        settings["dpo_repo"],
    ]
    if PUSH_TO_HUB
    else []
)

dry_run_command = [
    "python",
    "scripts/train_experiment.py",
    "--dataset-label",
    settings["label"],
    "--dataset-dir",
    settings["local_dir"],
    "--dataset-repo",
    settings["repo"],
    "--mlflow-ngrok",
    "--mlflow-ui-port",
    str(MLFLOW_UI_PORT),
    "--dry-run",
    *max_steps_arg,
    *max_samples_arg,
    *push_args,
]
subprocess.run(dry_run_command, check=True)

## Run training with live MLflow UI

This starts `mlflow ui`, opens an ngrok tunnel, prints the public MLflow URL, then trains SFT followed by DPO. Leave this cell running until training finishes.

In [ ]:
train_command = [arg for arg in dry_run_command if arg != "--dry-run"]
subprocess.run(train_command, check=True)

## Preserve outputs

Kaggle working storage is temporary. Zip MLflow runs and adapters before leaving the session.

In [ ]:
archive_name = f"{settings['label']}-training-artifacts.zip"
!zip -qr "$archive_name" mlruns outputs/experiments
print("wrote", archive_name)